# trainer-class-skeleton — ex1: minimal Trainer class: fit, validate, _step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `trainer-class-skeleton`. Running the final beacon cell reports progress against the `Trainer: Trainer class skeleton` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: Trainer class skeleton` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-class-skeleton`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-class-skeleton"
DD_SUBTOPIC = "Trainer: Trainer class skeleton"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Trainer class skeleton — quick refresher

A Trainer class is a thin object that owns the train/eval loop scaffold. It separates the GENERIC plumbing (epoch loop, optimizer step cycle, eval branch) from the MODEL-SPECIFIC `_step` body. The minimal interface looks like:

```
class Trainer:
    def __init__(self, model, optimizer, train_loader, val_loader):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.step = 0          # global step counter

    def _step(self, x, y):       # one batch → scalar loss
        ...

    def fit(self, n_epochs):
        for epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
            self.validate()

    def validate(self):
        self.model.eval()
        with t.inference_mode():
            ...
```

ARENA's chapter-3 training code follows exactly this shape (without PyTorch Lightning) — the same skeleton scales from a 3-line regression demo to a transformer training run.

### Exercise 1 — minimal Trainer class: fit, validate, _step

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the canonical Trainer-class skeleton — `__init__`, `_step`, `fit`, `validate` — so the per-batch training logic is separated from the model-specific loss computation.
> Keywords: trainer, fit-loop, validate, object-oriented
> ```

**KCs targeted:** `trainer-fit-loop-walks-epochs`, `trainer-validate-uses-eval-mode`

Implement `Ex1Trainer`. The minimal-but-correct Trainer skeleton.

1. `__init__(self, model, optimizer, train_loader, val_loader, loss_fn)`: store all five as attributes. Also initialize `self.step = 0` (global step counter, batches completed) and `self.history = {'train_loss': [], 'val_loss': []}`.

2. `_step(self, x, y) -> Tensor`: forward + loss only. Compute `logits = self.model(x)` then return `self.loss_fn(logits, y)`. Do NOT call `backward()` here.

3. `fit(self, n_epochs)`: for each epoch:
   - call `self.model.train()` (sets BN/dropout to train mode).
   - iterate `(x, y) in self.train_loader`:
     - `loss = self._step(x, y)`
     - `loss.backward()`
     - `self.optimizer.step()`
     - `self.optimizer.zero_grad()`
     - `self.step += 1`
     - `self.history['train_loss'].append(loss.item())`
   - call `self.validate()` at the end of each epoch.

4. `validate(self)`: switch to eval mode, run inference under `t.inference_mode()`, accumulate loss over `self.val_loader`, average, append to `self.history['val_loss']`.

The test uses a tiny linear regression task so you can check the loop walks the data, validates each epoch, and reduces loss across epochs.

In [ ]:
class Ex1Trainer:
    """Minimal Trainer class: model + optimizer + loaders + loss."""

    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        raise NotImplementedError()

    def _step(self, x, y):
        raise NotImplementedError()

    def fit(self, n_epochs: int):
        raise NotImplementedError()

    def validate(self):
        raise NotImplementedError()


def _test_ex1():
    from torch.utils.data import TensorDataset, DataLoader

    # Tiny linear regression task: y = 2x + 1 with noise.
    t.manual_seed(0)
    N = 64
    x_train = t.randn(N, 1)
    y_train = 2.0 * x_train + 1.0 + 0.05 * t.randn(N, 1)
    x_val = t.randn(16, 1)
    y_val = 2.0 * x_val + 1.0

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=8, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=8, shuffle=False)

    model = t.nn.Linear(1, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = t.nn.MSELoss()

    trainer = Ex1Trainer(model, opt, train_loader, val_loader, loss_fn)

    # === Attribute check ===
    assert trainer.model is model
    assert trainer.optimizer is opt
    assert trainer.train_loader is train_loader
    assert trainer.val_loader is val_loader
    assert trainer.loss_fn is loss_fn
    assert trainer.step == 0, f'step must start at 0, got {trainer.step}'
    assert 'train_loss' in trainer.history and 'val_loss' in trainer.history
    assert trainer.history['train_loss'] == []
    assert trainer.history['val_loss'] == []

    # === _step returns a SCALAR loss tensor (not a number, not yet detached) ===
    x_b, y_b = next(iter(train_loader))
    loss = trainer._step(x_b, y_b)
    assert isinstance(loss, t.Tensor), f'_step must return a tensor, got {type(loss)}'
    assert loss.ndim == 0, f'_step must return a scalar tensor, got shape {tuple(loss.shape)}'
    assert loss.requires_grad, '_step must return a grad-tracking loss (no .item() / .detach())'

    # === fit runs without error and steps the counter ===
    trainer.fit(n_epochs=3)
    expected_steps = 3 * len(train_loader)
    assert trainer.step == expected_steps, (
        f'after 3 epochs of {len(train_loader)} batches, step should be {expected_steps}; got {trainer.step}'
    )
    assert len(trainer.history['train_loss']) == expected_steps, (
        f'train_loss history length should equal step count'
    )
    assert len(trainer.history['val_loss']) == 3, (
        f'val_loss should be logged once per epoch; got {len(trainer.history["val_loss"])} entries after 3 epochs'
    )

    # === Loss decreases across epochs ===
    v0, v1, v2 = trainer.history['val_loss']
    assert v2 < v0, f'val loss should decrease over 3 epochs: epoch0={v0:.4f}, epoch2={v2:.4f}'
    # Model fitted close to truth.
    assert abs(model.weight.item() - 2.0) < 0.2, (
        f'weight should approach 2.0; got {model.weight.item():.4f}'
    )
    assert abs(model.bias.item() - 1.0) < 0.2, (
        f'bias should approach 1.0; got {model.bias.item():.4f}'
    )

    # === Validate uses eval mode + inference_mode (no grads tracked) ===
    # Hard to test directly; instead verify that after fit(), val_loss values are scalars (not tensors).
    for v in trainer.history['val_loss']:
        assert isinstance(v, float), (
            f'val_loss entries should be Python floats (use .item()); got {type(v)}'
        )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Ex1Trainer:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        logits = self.model(x)
        return self.loss_fn(logits, y)

    def fit(self, n_epochs):
        for _epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total = 0.0
        count = 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)
```

**Why separate `_step` from `fit`.** The model-specific logic (forward + loss) lives in `_step`. The generic scaffolding (epoch loop, optimizer cycle, validation) lives in `fit`. Swapping models means rewriting only `_step`. PyTorch Lightning generalizes this further with `training_step` / `validation_step` hooks — same idea, more ceremony.

**Why `self.history` is a dict.** As you add more metrics (accuracy, learning rate, gradient norms) the history dict just grows new keys. A `(train_losses, val_losses)` tuple would force a refactor.

**Why `inference_mode()` inside `validate`.** Eval-only code shouldn't build the autograd graph (wastes memory and time). `inference_mode` is stricter than `no_grad` — it also disables version counters. Both are correct; `inference_mode` is what new PyTorch code uses.

**Weighted-average val loss.** Multiplying by `x.shape[0]` (the batch size) handles the partial last batch correctly. A naive `mean(losses)` would over-weight the small partial batch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()